# 第 1 章:大图景 —— LLM 端到端是什么

欢迎来到 **MiniMind 教程**。这套教程的目标只有一个:让你**真正理解一个 LLM 的每一行代码**,从分词器到强化学习对齐。

本章我们先不写代码,而是**先把 minimind 跑起来**,亲眼看看一个 LLM 在推理时发生了什么,然后把整个过程拆成你将在后续 14 章学到的零件。

## 1.1 为什么要「从零开始」?

今天的 LLM 生态有一个矛盾:

- 一方面,`transformers` / `trl` / `peft` 这些框架让你**十几行代码**就能完成「加载模型 + 训练 + 推理」的全流程。
- 另一方面,这种高效封装**把你和底层实现隔开了** —— 你知道怎么调用,但不知道为什么这样写。

> 这就像用乐高搭飞机 vs 坐头等舱。minimind 的哲学是:**亲手用乐高搭一架飞机,远比坐头等舱更令人兴奋。**(README_en.md:79)

minimind 的设计参数:

| 项目 | 值 | 对比 GPT-3 |
|---|---|---|
| 参数量 | ~64M | 1/2700 |
| 训练成本 | ~¥3(单张 3090,2 小时) | — |
| 模型代码 | **344 行**(一个文件) | — |
| 训练算法 | 全部手写,不依赖 trl/peft 封装 | — |

**目标(Goal)**:学完 15 章后,你能从头写出 minimind 的每一个组件,并理解每行代码的用意。

## 1.2 一个 LLM 的完整生命周期

在拆解之前,先看全貌。一个现代 LLM(如 Qwen3 / DeepSeek / minimind)从「什么都不会」到「能对话、能用工具」需要经过这些阶段:

```
  原始文本
    │
    ▼
  ① 分词器(Tokenizer)── 文本 → token id 序列
    │
    ▼
  ② 模型架构(Model)── Transformer:embed → attention → FFN → lm_head
    │
    ▼
  ③ 预训练(Pretrain)── 海量文本上做「下一个 token 预测」
    │              → 模型学会语言的统计规律
    ▼
  ④ 监督微调(SFT)── 高质量对话数据上做「指令遵循」
    │              → 模型学会扮演 assistant
    ▼
  ⑤ 对齐(Alignment)── DPO / PPO / GRPO / Agent-RL
    │              → 模型变得「有用、诚实、无害」
    ▼
  ⑥ 推理部署(Inference)── API 服务 / 流式输出 / 工具调用
```

**minimind 覆盖全部 6 个阶段。** 这也是为什么我们用 15 章来教它。

| minimind 阶段 | 对应代码 | 本教程章节 |
|---|---|---|
| ① 分词器 | `train_tokenizer.py` | 第 2 章 |
| ② 模型架构 | `model_minimind.py`(344 行) | 第 3-7 章 |
| ③ 预训练 | `train_pretrain.py` | 第 8 章 |
| ④ SFT | `train_full_sft.py` + `train_lora.py` | 第 9-10 章 |
| ⑤ 对齐 | `train_dpo.py` / `train_ppo.py` / `train_grpo.py` / `train_agent.py` | 第 12-15 章 |
| ⑥ 推理 | `eval_llm.py` + `serve_openai_api.py` | 第 11 章 + **本章** |

&nbsp;

---

## 1.3 先跑起来:加载 minimind 做一次推理

**最快的去神秘化方式:先把模型跑起来,再看它内部做了什么。**

> ⚠️ 本章的代码需要你在自己的环境里运行(需要 GPU 或耐心用 CPU)。如果还没有权重,先按 `setup/README.md` 下载预训练权重。

### 1.3.1 最简单的方式:命令行

minimind 提供了一个 98 行的推理脚本 `eval_llm.py`(本章拆解的对象):

In [ ]:
# 在 minimind 根目录运行
# cd /home/minimind
# python eval_llm.py --weight full_sft --load_from model
#
# 然后选择 [0] 自动测试,你会看到:
# 💬: 你有什么特长?
# 🧠: 我是一个语言模型,我的特长是理解和生成文本...(流式输出)
# [Speed]: 45.32 tokens/s

### 1.3.2 用代码加载(本章后续拆解的基础)

那行命令背后发生了什么?让我们用 Python 代码一步步还原。先看模型的「身份证」—— 配置:

In [ ]:
import torch
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

# minimind 的配置:64M 参数,8 层,768 维,6400 词表
config = MiniMindConfig(
    hidden_size=768,
    num_hidden_layers=8,
    # vocab_size=6400, num_attention_heads=8, num_key_value_heads=4, ...
    # 详见第 3 章
)
print(f"词表大小: {config.vocab_size}")
print(f"层数: {config.num_hidden_layers}")
print(f"隐藏维度: {config.hidden_size}")
# 词表大小: 6400
# 层数: 8
# 隐藏维度: 768

In [ ]:
# 加载预训练权重(需要先下载,见 setup/README.md)
model = MiniMindForCausalLM(config)
# ckp_path = './out/full_sft_768.pth'
# model.load_state_dict(torch.load(ckp_path, map_location='cpu'), strict=True)
model = model.half().eval()  # 推理:半精度省显存
# ⚠️ 训练时不要 .half()!用 float32 权重 + autocast(bfloat16),否则 grad 溢出 NaN

# 看看参数量
n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params / 1e6:.1f}M")  # 模型参数量: ~26-64M

## 1.4 拆解推理过程:到底发生了什么?

现在模型加载好了。一个完整的「推理」可以拆成 **3 步 + 1 个对话模板**:

### 1.4.1 输入:文本如何变成 token id

LLM 不认识文字,只认识数字。**分词器(tokenizer)** 负责把文本切成 token(子词)再映射成整数 id。

minimind 用的是 **BPE + ByteLevel** 分词器(第 2 章详解),词表只有 6400:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('./model')

text = "你好,minimind!"
token_ids = tokenizer.encode(text)
print(f"原文: {text}")
print(f"token ids: {token_ids}")
print(f"tokens: {tokenizer.convert_ids_to_tokens(token_ids)}")
# token ids: [1, 5310, 2863, 293, 15, 116, 18750, 18819, 325, 2]
# tokens: ['<|im_start|>', '你好', ',', 'min', 'im', 'ind', '!', '<|im_end|>']

# 注意首尾的特殊 token:
#   <|im_start|> (id=1) = bos,标记对话开始
#   <|im_end|>   (id=2) = eos,标记对话结束

> **为什么是 6400?** GPT-4 的词表约 10 万。minimind 故意用极小词表,让 embedding 层(参数量 = vocab × dim = 6400 × 768 ≈ 490 万)在 64M 总参数中占比合理。这是一个**工程取舍**,第 3 章会详细讨论。

### 1.4.2 模型:从 token id 到 logits

token id 序列进入模型后,经过一条固定的张量流水线:

```
input_ids [B, T]          # (batch, sequence_length)
  │
  ▼ embed_tokens            # 查表:id → 向量
hidden_states [B, T, 768]  # 每个 token 变成 768 维向量
  │
  ▼ × 8 个 Transformer Block
  │   每个 block:
  │   ┌─ RMSNorm ─→ Attention(自注意力 + 残差)
  │   └─ RMSNorm ─→ FeedForward(SwiGLU + 残差)
hidden_states [B, T, 768]
  │
  ▼ final RMSNorm
  ▼ lm_head (线性层)       # 768 维 → 6400 维(词表大小)
logits [B, T, 6400]        # 每个位置对每个词的原始分数
```

**logits** 是模型对「下一个 token 是什么」的打分。第 4-6 章会逐层拆解这条流水线。

### 1.4.3 输出:从 logits 到文字(采样)

拿到 logits 后,**生成(generate)** 过程决定下一个 token 选哪个。minimind 手写了一个采样器(`model_minimind.py:~314-345` (@67f114a),第 7 章详解):

In [ ]:
prompt = "你有什么特长?"
# 先套上对话模板(见 1.4.4)
conversation = [{"role": "user", "content": prompt}]
inputs_text = tokenizer.apply_chat_template(
    conversation, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(inputs_text, return_tensors='pt')
print(f"输入文本(含模板): {repr(inputs_text[:80])}...")
# 输入文本(含模板): '<|im_start|>user\n你有什么特长?<|im_end|>\n<|im_start|>assistant\n'...

In [ ]:
# 生成!这是 eval_llm.py 第 ~86-90 行的核心调用 (@67f114a)
with torch.no_grad():
    generated_ids = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=128,
        do_sample=True,
        temperature=0.85,      # 控制随机性,越大越随机
        top_p=0.95,            # nucleus 采样,只在累积概率 95% 的词里选
        repetition_penalty=1,  # 重复惩罚
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# 解码输出(去掉输入部分)
response = tokenizer.decode(
    generated_ids[0][len(inputs['input_ids'][0]):],
    skip_special_tokens=True
)
print(f"模型回复: {response}")
# 模型回复: 我是一个语言模型,擅长理解和生成文本...

采样的三个关键旋钮(第 7 章逐个推导):

- **temperature**:把 logits 除以这个值再做 softmax。`< 1` 更确定(倾向高分词),`> 1` 更随机。minimind 默认 `0.85`。
- **top_p**(nucleus sampling):把词按概率从高到低排序,只保留累积概率达到 `p` 的那些词,其余直接丢弃。避免极低概率的「胡言乱语」。
- **repetition_penalty**:已出现的 token 分数打折扣,防止「复读机」。

> **KV cache**:生成第 N 个 token 时,前 N-1 个 token 的 Key/Value 已经算过了。minimind 把它们缓存起来,每步只算新 token 的 K/V —— 这是自回归生成能跑快的关键。第 7 章详解。

### 1.4.4 对话模板:多轮对话如何表示

minimind 用 ChatML 风格的对话模板(`tokenizer_config.json` 里的 Jinja2 模板,第 2 章详解):

```
<|im_start|>user
你有什么特长?<|im_end|>
<|im_start|>assistant
我是一个语言模型...<|im_end|>
<|im_start|>user
那你能写代码吗?<|im_end|>
<|im_start|>assistant
```

每个角色用 `<|im_start|>role` 开头,`<im_end|>` 结尾。`add_generation_prompt=True` 时会在末尾留一个 `<|im_start|>assistant\n`,让模型从那里开始续写。

这套模板在 SFT 阶段(第 9 章)被用来构造训练数据,在推理时(本章)被用来格式化用户输入。**理解它对后续章节至关重要。**

&nbsp;

---

## 1.5 本教程的 15 章路线图

现在你已经见过全貌了。接下来 14 章会**由内向外**逐层拆解:

### 第一部分:模型半(ch2-7)

把 344 行的 `model_minimind.py` 拆成 6 章,从最底层的原子(RMSNorm、RoPE)一直搭到完整的生成循环。

| 章 | 学什么 | 对应行号 |
|---|---|---|
| 2 | 分词器:BPE 怎么工作 | `train_tokenizer.py` |
| 3 | 配置:为什么 dim=768, layers=8 | `model_minimind.py:~10-50` (@67f114a) |
| 4 | 注意力:RMSNorm + RoPE + GQA | `model_minimind.py:~55-139` (@67f114a) |
| 5 | FFN 与 Block:SwiGLU + 残差 | `model_minimind.py:~141-208` (@67f114a) |
| 6 | 模型组装:embed → 堆叠 → lm_head | `model_minimind.py:~210-312` (@67f114a) |
| 7 | 生成:KV cache + 采样循环 | `model_minimind.py:~314-345` (@67f114a) |

### 第二部分:训练半(ch8-11)

模型搭好后,怎么训练?这部分覆盖「把权重从随机变成能用」的全过程。

| 章 | 学什么 |
|---|---|
| 8 | 预训练循环:AdamW + 余弦 LR + AMP + DDP |
| 9 | SFT:answer-only loss masking(教模型扮演 assistant 的关键) |
| 10 | LoRA:低秩适配,只改 0.5% 参数就能微调 |
| 11 | 推理工程:FastAPI 服务 + 工具调用解析 |

### 第三部分:对齐 / RL 半(ch12-15)

**这是 minimind 相对其他「从零教程」最大的差异化** —— 完整覆盖现代 LLM 的对齐技术栈。

| 章 | 学什么 |
|---|---|
| 12 | DPO:从人类偏好数据直接学习 |
| 13 | PPO / GRPO / CISPO:统一 PO 框架(三项可替换) |
| 14 | Agent-RL:多轮工具调用作为强化学习轨迹 |
| 15 | 知识蒸馏 + MoE:白盒蒸馏与混合专家 |

&nbsp;

---

## 1.6 环境准备

在进入第 2 章之前,请确保:

- [ ] 已 clone minimind(`git clone https://github.com/rickqi/minimind.git`)
- [ ] 已安装 Python 3.10+ 和依赖(`pip install -r requirements.txt`)
- [ ] (推荐)有一张 8GB+ 显存的 GPU
- [ ] (可选)已下载预训练权重到 `out/` 目录

详细步骤见 [`setup/README.md`](../../setup/README.md)。

## Summary and takeaways

恭喜!你已经:

- 理解了一个 LLM 的**完整生命周期**(分词 → 模型 → 预训练 → SFT → 对齐 → 推理)
- 亲手**加载并运行**了 minimind,看到它生成文字
- 拆解了推理的 **4 个零件**:tokenizer → model → sampling → chat_template
- 看到了接下来 **14 章的路线图**

> **核心认知**:LLM 不是魔法。它是一个「文本进、文本出」的函数,内部是大量矩阵乘法和 softmax。后面 14 章会把每一层都拆开给你看。

- 精简复习版见 [`./big-picture.ipynb`](./big-picture.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

下一章:[第 2 章 · 分词器](../ch02/01_main-chapter-code/README.md)